## Overview
This notebook builds a **single-agent smart assistant** that routes a user query to the right tool (calculator, keyword extractor, text stats, case converter) or falls back to a general response — always returning structured JSON with `type` and `result`.


# Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---


### Step 1: Calculator Tool
Defines a helper function that safely evaluates a math expression string and returns `"Error in calculation"` instead of crashing if the expression is invalid.


In [1]:
# TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

**Observation:** `calculator()` is defined and ready to use. It wraps `eval()` in a try/except so bad expressions return a safe error string rather than raising an exception.


###  Step 2: Keyword Extractor Tool
Defines a helper function that splits input text into words and returns up to 5 unique lowercase words longer than 4 characters as keywords.


In [2]:
# TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

**Observation:** `extract_keywords()` is defined and ready to use. It filters short/common words out by length and de-duplicates, returning at most 5 keywords.


###  Step 3 (Bonus): Text Statistics Tool
Defines a helper function that computes basic statistics — word count, character count, and sentence count — for a piece of text.


In [3]:
# TOOL 3: Text Statistics
# (Bonus tool #1)

def text_stats(text: str) -> dict:
    """Return basic statistics about a piece of text."""
    try:
        words = text.split()
        sentences = [s for s in text.replace("!", ".").replace("?", ".").split(".") if s.strip()]
        return {
            "word_count": len(words),
            "char_count": len(text),
            "sentence_count": len(sentences)
        }
    except Exception:
        return {"error": "Could not compute text statistics"}


**Observation:** `text_stats()` is defined and ready to use. It returns a dictionary of counts, or an `"error"` key if the input can't be processed.


###  Step 4 (Bonus): Case Converter Tool
Defines a helper function that converts text to upper, lower, or title case depending on the `mode` argument.


In [4]:
#  TOOL 4: Case Converter
# (Bonus tool #2)

def convert_case(text: str, mode: str = "upper") -> str:
    """Convert text to upper, lower, or title case."""
    try:
        if mode == "upper":
            return text.upper()
        elif mode == "lower":
            return text.lower()
        elif mode == "title":
            return text.title()
        else:
            return text
    except Exception:
        return "Error in case conversion"


**Observation:** `convert_case()` is defined and ready to use. It supports `"upper"`, `"lower"`, and `"title"` modes and falls back safely on errors.


### Step 5 : Logging Setup
Configures a `smart_agent` logger so every routing decision and error the agent makes is printed with a timestamp — useful for debugging and for verifying the routing logic works.


In [5]:
#  Logging Setup
# (structured logging for every agent decision)

import logging

logger = logging.getLogger("smart_agent")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
    handler.setFormatter(formatter)
    logger.addHandler(handler)


**Observation:** The logger is configured and attached to a stream handler. No output yet — logs will appear once `agent()` is called below.


### Step 6: Agent Function — Routing Logic
Implements the core router: it lowercases the query, matches it against regex patterns (with word boundaries and synonyms) to pick a route — calculation, keywords, stats, case, or general fallback — calls the matching tool, logs the decision, and always returns a `{"type": ..., "result": ...}` dictionary, catching any unexpected errors along the way.


In [6]:
#  AGENT FUNCTION (IMPLEMENTED — improved routing + logging + more tools)

import re
import json

# Routing table: each entry is (route_name, [regex patterns], handler)
# Patterns use word boundaries so "calculate" doesn\'t accidentally match
# inside unrelated words, and synonyms are supported.
ROUTES = [
    ("calculation", [r"\bcalculate\b", r"\bcompute\b", r"\bwhat is\b.*[\d]"]),
    ("keywords",    [r"\bkeywords?\b", r"\bextract\b"]),
    ("stats",       [r"\bstats?\b", r"\bword count\b", r"\bcount words\b"]),
    ("case",        [r"\bupper\s*case\b", r"\blower\s*case\b", r"\btitle\s*case\b", r"\bconvert.*case\b"]),
]


def _match_route(query_lower: str):
    """Return the first matching route name, or None if nothing matches."""
    for route_name, patterns in ROUTES:
        for pattern in patterns:
            if re.search(pattern, query_lower):
                return route_name
    return None


def agent(query: str):
    """
    Single-agent task router with improved (regex, word-boundary, synonym-aware)
    routing, structured logging of every decision, and four tools:
    calculator, keyword extractor, text stats, and case converter.
    Always returns a JSON-safe dict with keys "type" and "result".
    """
    logger.info(f"Received query: {query!r}")

    try:
        if not isinstance(query, str) or not query.strip():
            logger.error("Empty or invalid query received")
            return {"type": "error", "result": "Empty or invalid query"}

        query_lower = query.lower()
        route = _match_route(query_lower)
        logger.info(f"Routed to: {route or 'general'}")

        # --- Route: Calculation ---
        if route == "calculation":
            after_keyword = re.split(r"calculate|compute", query_lower, maxsplit=1)[-1].strip()

            if not after_keyword:
                logger.error("No expression found after calculation keyword")
                return {"type": "error", "result": "No valid mathematical expression found in query"}

            if re.search(r"[a-zA-Z]", after_keyword):
                logger.error(f"Malformed expression (letters present): {after_keyword!r}")
                return {"type": "error", "result": f"Malformed mathematical expression: '{after_keyword}'"}

            expression = re.sub(r"[^0-9\.\+\-\*\/\(\)\s]", "", after_keyword).strip()

            if not expression:
                logger.error("Expression empty after sanitization")
                return {"type": "error", "result": "No valid mathematical expression found in query"}

            calc_result = calculator(expression)

            if calc_result == "Error in calculation":
                logger.error(f"Calculator failed on expression: {expression!r}")
                return {"type": "error", "result": f"Could not evaluate expression: '{expression}'"}

            logger.info(f"Calculation success: {expression} = {calc_result}")
            return {"type": "calculation", "result": calc_result}

        # --- Route: Keyword extraction ---
        elif route == "keywords":
            keywords = extract_keywords(query)

            if not keywords:
                logger.error("No keywords extracted")
                return {"type": "error", "result": "No keywords could be extracted from query"}

            logger.info(f"Keywords extracted: {keywords}")
            return {"type": "keywords", "result": keywords}

        # --- Route: Text statistics (bonus tool) ---
        elif route == "stats":
            stats = text_stats(query)

            if "error" in stats:
                logger.error("Text stats computation failed")
                return {"type": "error", "result": stats["error"]}

            logger.info(f"Text stats computed: {stats}")
            return {"type": "stats", "result": stats}

        # --- Route: Case conversion (bonus tool) ---
        elif route == "case":
            if "upper" in query_lower:
                mode = "upper"
            elif "lower" in query_lower:
                mode = "lower"
            else:
                mode = "title"

            converted = convert_case(query, mode)
            logger.info(f"Case conversion ({mode}): {converted}")
            return {"type": "case", "result": converted}

        # --- Route: Fallback / general conversational queries ---
        else:
            logger.info("No specific route matched, using general fallback")
            return {
                "type": "general",
                "result": f"I received your query: \"{query}\". No specific tool matched, so this is a general response."
            }

    # --- Safety net: never let the agent crash the pipeline ---
    except Exception as e:
        logger.error(f"Unexpected error: {str(e)}")
        return {"type": "error", "result": f"Unexpected error while processing query: {str(e)}"}


**Observation:** `agent()` is defined. It combines all four tools behind one routing function and guarantees a JSON-safe response even on malformed input.


##  Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

###  Step 7: Test Cases
Runs the agent against 8 sample queries covering every route — calculation, keywords, general fallback, division-by-zero error, malformed-expression error, stats, and case conversion — and asserts each response has the required `type`/`result` keys.


In [7]:
#  Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Calculate 10 / 0",
    "Calculate banana + 5",
    "keywords",
    "Give me word count stats for this sentence",
    "Convert this to uppercase please"
]

for q in queries:
    response = agent(q)
    print("Query:", q)
    print("Response (JSON):", json.dumps(response, indent=2))
    assert isinstance(response, dict), "Agent must return a dict"
    assert "type" in response and "result" in response, "Response missing required keys"
    print("-" * 50)


13:04:09 | INFO | Received query: 'Calculate 20 + 5'
INFO:smart_agent:Received query: 'Calculate 20 + 5'
13:04:09 | INFO | Routed to: calculation
INFO:smart_agent:Routed to: calculation
13:04:09 | INFO | Calculation success: 20 + 5 = 25
INFO:smart_agent:Calculation success: 20 + 5 = 25
13:04:09 | INFO | Received query: 'Extract keywords from Artificial Intelligence is transforming industries'
INFO:smart_agent:Received query: 'Extract keywords from Artificial Intelligence is transforming industries'
13:04:09 | INFO | Routed to: keywords
INFO:smart_agent:Routed to: keywords
13:04:09 | INFO | Keywords extracted: ['keywords', 'transforming', 'intelligence', 'industries', 'extract']
INFO:smart_agent:Keywords extracted: ['keywords', 'transforming', 'intelligence', 'industries', 'extract']
13:04:09 | INFO | Received query: 'What is machine learning?'
INFO:smart_agent:Received query: 'What is machine learning?'
13:04:09 | INFO | Routed to: general
INFO:smart_agent:Routed to: general
13:04:09 |

Query: Calculate 20 + 5
Response (JSON): {
  "type": "calculation",
  "result": "25"
}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response (JSON): {
  "type": "keywords",
  "result": [
    "keywords",
    "transforming",
    "intelligence",
    "industries",
    "extract"
  ]
}
--------------------------------------------------
Query: What is machine learning?
Response (JSON): {
  "type": "general",
  "result": "I received your query: \"What is machine learning?\". No specific tool matched, so this is a general response."
}
--------------------------------------------------
Query: Calculate 10 / 0
Response (JSON): {
  "type": "error",
  "result": "Could not evaluate expression: '10 / 0'"
}
--------------------------------------------------
Query: Calculate banana + 5
Response (JSON): {
  "type": "error",
  "result": "Malformed mathematical expression: 'banana + 5'"
}
--------------------------------

**Observation:** All 8 test queries returned valid JSON with both keys. Routing worked correctly: `"20 + 5"` → `25`, keyword extraction and the general fallback behaved as expected, `10 / 0` and `"banana + 5"` were correctly caught as `"error"`, and the new `stats`/`case` tools returned correct results — confirming the routing, error handling, and bonus tools all work together.


###  Step 8: Interactive Mode
Lets you type queries one at a time and see the agent's JSON response and log lines live, until you type `exit`.


In [8]:
# Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    response = agent(user_input)
    print("Response (JSON):", json.dumps(response, indent=2))


Enter query (type 'exit' to stop): Calculate 12 * 8


13:04:32 | INFO | Received query: 'Calculate 12 * 8'
INFO:smart_agent:Received query: 'Calculate 12 * 8'
13:04:32 | INFO | Routed to: calculation
INFO:smart_agent:Routed to: calculation
13:04:32 | INFO | Calculation success: 12 * 8 = 96
INFO:smart_agent:Calculation success: 12 * 8 = 96


Response (JSON): {
  "type": "calculation",
  "result": "96"
}
Enter query (type 'exit' to stop): Convert this sentence to uppercase rutuja


13:04:42 | INFO | Received query: 'Convert this sentence to uppercase rutuja'
INFO:smart_agent:Received query: 'Convert this sentence to uppercase rutuja'
13:04:42 | INFO | Routed to: case
INFO:smart_agent:Routed to: case
13:04:42 | INFO | Case conversion (upper): CONVERT THIS SENTENCE TO UPPERCASE RUTUJA
INFO:smart_agent:Case conversion (upper): CONVERT THIS SENTENCE TO UPPERCASE RUTUJA


Response (JSON): {
  "type": "case",
  "result": "CONVERT THIS SENTENCE TO UPPERCASE RUTUJA"
}
Enter query (type 'exit' to stop): exit


**Observation:** Manual test run confirms it works live — `"Calculate 12 * 8"` correctly routed to calculation and returned `96`, and `"Convert this sentence to uppercase rutuja"` correctly routed to case conversion and returned the uppercased text, before exiting on `"exit"`.
